# Shortcut Learning Mitigation Experiments For Subject-Independent EEG


Terminology:

- **ERM Transformer**: the standard vanilla Transformer trained normally.
- **JTT**: Just Train Twice. Train once, identify misclassified training examples, then train a second model with those examples upweighted. It does not require group labels.
- **GroupDRO**: train a model to minimize worst-group loss. Here the oracle group is `diagnosis x subject`, so it assumes subject/group labels are available.
- **DFR**: Deep Feature Reweighting / last-layer retraining. Freeze the ERM feature extractor and retrain only the final classifier on a balanced reweighting set.

References:

- JTT: Liu et al., **Just Train Twice: Improving Group Robustness without Training Group Information**, ICML 2021. https://proceedings.mlr.press/v139/liu21f.html
- GroupDRO: Sagawa et al., **Distributionally Robust Neural Networks for Group Shifts**, ICLR 2020. https://openreview.net/forum?id=ryxGuJrFvS
- DFR: Kirichenko, Izmailov, Wilson, **Last Layer Re-Training is Sufficient for Robustness to Spurious Correlations**, ICLR 2023 / arXiv 2022. https://arxiv.org/abs/2204.02937

All experiments use  raw EEG samples:

```text
1-second non-overlapping windows
subject-independent 60/20/20 split by subject
sample-level accuracy / macro precision / macro recall / macro F1
```


In [ ]:
 # ! pip install mne

In [ ]:
!pip install awscli
! aws s3 sync --no-sign-request s3://openneuro.org/ds004504 ds004504-download/

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: rsa
    Found existing installation: rsa 4.9.1
    Uninstalling rsa-4.9.1:
      Successfully uninstalled rsa-4.9.1
  Attempting uninstall: docutils
    Found existing installation: docutils 0.21.2
    Uninstalling docutils-0.21.2:
      Successfully uninstalled docutils-0.21.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.
download: s3://openneuro.org/ds004504/dataset_description.json to ds004504-download/dataset_description.json
down

In [ ]:
import os
import numpy as np
import pandas as pd
import mne

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import plotly.express as px

RANDOM_STATE = 42
TORCH_SEED = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(TORCH_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cuda


## Dataset Setup


In [ ]:
DATASET_DIR = "/content/ds004504-download"
DERIVATIVES_DIR = os.path.join(DATASET_DIR, "derivatives")
PARTICIPANTS_FILE = os.path.join(DATASET_DIR, "participants.tsv")

participants = pd.read_csv(PARTICIPANTS_FILE, sep="\t")
label_map = {
    "A": "AD",
    "F": "FTD",
    "C": "CN",
    "AD": "AD",
    "FTD": "FTD",
    "CN": "CN",
}
participants["label"] = participants["Group"].map(label_map)
participants.head()


,participant_id,Gender,Age,Group,MMSE,label
0,sub-001,F,57,A,16,AD
1,sub-002,F,78,A,22,AD
2,sub-003,M,70,A,14,AD
3,sub-004,F,67,A,20,AD
4,sub-005,M,70,A,22,AD


In [ ]:
def load_one_subject_raw_samples(file_path, subject_id, label, target_sfreq=256, sample_seconds=1.0):
    raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)
    raw.pick_types(eeg=True)

    if raw.info["sfreq"] != target_sfreq:
        raw.resample(target_sfreq, npad="auto", verbose=False)

    data = raw.get_data().astype(np.float32) * 1e6
    sample_len = int(target_sfreq * sample_seconds)
    n_samples = data.shape[1] // sample_len
    data = data[:, :n_samples * sample_len]

    samples = data.reshape(data.shape[0], n_samples, sample_len)
    samples = np.transpose(samples, (1, 2, 0))
    labels = np.repeat(label, n_samples)
    subjects = np.repeat(subject_id, n_samples)
    return samples, labels, subjects


def build_raw_eeg_sample_dataset(participants_df, max_subjects=None):
    all_samples = []
    all_labels = []
    all_subjects = []

    rows = list(participants_df.iterrows())
    if max_subjects is not None:
        rows = rows[:max_subjects]

    for _, row in rows:
        subject_id = row["participant_id"]
        label = row["label"]
        eeg_file = os.path.join(DERIVATIVES_DIR, subject_id, "eeg", f"{subject_id}_task-eyesclosed_eeg.set")

        if not os.path.exists(eeg_file):
            print(f"Missing file: {eeg_file}")
            continue

        print(f"Loading {subject_id} | {label}")
        try:
            samples, labels, subjects = load_one_subject_raw_samples(eeg_file, subject_id, label)
            all_samples.append(samples)
            all_labels.append(labels)
            all_subjects.append(subjects)
        except Exception as e:
            print(f"Error loading {subject_id}: {e}")

    X = np.concatenate(all_samples, axis=0).astype(np.float32)
    y_text = np.concatenate(all_labels, axis=0).astype(str)
    subjects = np.concatenate(all_subjects, axis=0).astype(str)
    return X, y_text, subjects


In [ ]:
X_raw, y_text_raw, subjects_raw = build_raw_eeg_sample_dataset(participants, max_subjects=None)

label_encoder = LabelEncoder()
y_raw = label_encoder.fit_transform(y_text_raw)
class_names = label_encoder.classes_

print("X_raw:", X_raw.shape)
print("classes:", list(class_names))
print("subjects:", len(np.unique(subjects_raw)))
print("samples:", len(y_raw))


Loading sub-001 | AD
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Loading sub-002 | AD
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Loading sub-003 | AD
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_3218/2052934808.py:2: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)


Loading sub-004 | AD
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_3218/2052934808.py:2: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)


Loading sub-005 | AD
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Loading sub-006 | AD
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


...

Loading sub-087 | FTD
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_3218/2052934808.py:2: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)


Loading sub-088 | FTD
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


/tmp/ipykernel_3218/2052934808.py:2: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose=False)


X_raw: (69794, 256, 19)
classes: [np.str_('AD'), np.str_('CN'), np.str_('FTD')]
subjects: 88
samples: 69794


In [ ]:
def make_subject_independent_indices(X, y, subjects, random_state=42):
    gss_train = GroupShuffleSplit(n_splits=1, test_size=0.4, random_state=random_state)
    train_idx, temp_idx = next(gss_train.split(X, y, groups=subjects))

    gss_val_test = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=random_state)
    val_rel_idx, test_rel_idx = next(gss_val_test.split(X[temp_idx], y[temp_idx], groups=subjects[temp_idx]))

    return train_idx, temp_idx[val_rel_idx], temp_idx[test_rel_idx]


def standardize_raw_by_train(X_train, X_val, X_test, eps=1e-6):
    mean = X_train.mean(axis=(0, 1), keepdims=True)
    std = X_train.std(axis=(0, 1), keepdims=True)
    std = np.maximum(std, eps)
    return (
        ((X_train - mean) / std).astype(np.float32),
        ((X_val - mean) / std).astype(np.float32),
        ((X_test - mean) / std).astype(np.float32),
    )

train_idx, val_idx, test_idx = make_subject_independent_indices(X_raw, y_raw, subjects_raw, RANDOM_STATE)

X_train, X_val, X_test = standardize_raw_by_train(X_raw[train_idx], X_raw[val_idx], X_raw[test_idx])
y_train, y_val, y_test = y_raw[train_idx], y_raw[val_idx], y_raw[test_idx]
subjects_train, subjects_val, subjects_test = subjects_raw[train_idx], subjects_raw[val_idx], subjects_raw[test_idx]

print("train/val/test samples:", len(train_idx), len(val_idx), len(test_idx))
print("train/val/test subjects:", len(np.unique(subjects_train)), len(np.unique(subjects_val)), len(np.unique(subjects_test)))


train/val/test samples: 40753 14421 14620
train/val/test subjects: 52 18 18


## Model And Training Helpers


In [ ]:
class EEGDataset(Dataset):
    def __init__(self, X, y, group=None, weight=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.group = None if group is None else torch.tensor(group, dtype=torch.long)
        self.weight = None if weight is None else torch.tensor(weight, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        items = [self.X[idx], self.y[idx]]
        if self.group is not None:
            items.append(self.group[idx])
        if self.weight is not None:
            items.append(self.weight[idx])
        return tuple(items)


def make_loader(X, y, group=None, weight=None, batch_size=128, shuffle=False):
    return DataLoader(EEGDataset(X, y, group=group, weight=weight), batch_size=batch_size, shuffle=shuffle)


In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class VanillaEEGTransformer(nn.Module):
    def __init__(self, n_channels, n_classes, d_model=128, n_heads=4, n_layers=3, d_ff=256, dropout=0.1):
        super().__init__()
        self.input_projection = nn.Linear(n_channels, d_model)
        self.positional_encoding = SinusoidalPositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, n_classes)

    def encode(self, x):
        x = self.input_projection(x)
        x = self.positional_encoding(x)
        x = self.encoder(x)
        return self.norm(x.mean(dim=1))

    def forward(self, x):
        return self.classifier(self.encode(x))


def new_model():
    return VanillaEEGTransformer(
        n_channels=X_train.shape[2],
        n_classes=len(class_names),
        d_model=128,
        n_heads=4,
        n_layers=3,
        d_ff=256,
        dropout=0.1,
    ).to(DEVICE)


In [ ]:
def metric_dict(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def evaluate_model(model, loader, title="model"):
    model.eval()
    preds = []
    targets = []
    with torch.no_grad():
        for batch in loader:
            batch_x, batch_y = batch[0], batch[1]
            logits = model(batch_x.to(DEVICE))
            preds.append(logits.argmax(dim=1).cpu().numpy())
            targets.append(batch_y.numpy())
    y_true = np.concatenate(targets)
    y_pred = np.concatenate(preds)
    metrics = metric_dict(y_true, y_pred)
    print(f"\n{title}")
    for key, value in metrics.items():
        print(f"{key}: {value:.4f}")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0))
    return metrics, y_true, y_pred


def load_transformer_checkpoint(path):
    checkpoint = torch.load(path, map_location=DEVICE, weights_only=False)
    config = checkpoint.get("model_config", {})
    model = VanillaEEGTransformer(
        n_channels=config.get("n_channels", X_train.shape[2]),
        n_classes=config.get("n_classes", len(class_names)),
        d_model=config.get("d_model", 128),
        n_heads=config.get("n_heads", 4),
        n_layers=config.get("n_layers", 3),
        d_ff=config.get("d_ff", 256),
        dropout=config.get("dropout", 0.1),
    ).to(DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model, checkpoint


def save_checkpoint(path, model, metrics, method):
    torch.save(
        {
            "model_name": "VanillaEEGTransformer",
            "method": method,
            "state_dict": model.state_dict(),
            "model_config": {
                "n_channels": X_train.shape[2],
                "n_classes": len(class_names),
                "d_model": 128,
                "n_heads": 4,
                "n_layers": 3,
                "d_ff": 256,
                "dropout": 0.1,
            },
            "class_names": list(class_names),
            "test_metrics": {k: float(v) for k, v in metrics.items()},
            "random_state": RANDOM_STATE,
        },
        path,
    )
    print("saved:", path)


In [ ]:
def train_erm(train_loader, val_loader, test_loader, title="ERM", epochs=30, lr=1e-4, patience=7, weight_decay=1e-4, weighted=False):
    model = new_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_f1 = -np.inf
    best_state = None
    stale = 0

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for batch in train_loader:
            batch_x, batch_y = batch[0].to(DEVICE), batch[1].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x)
            per_sample_loss = nn.functional.cross_entropy(logits, batch_y, reduction="none")

            if weighted:
                batch_w = batch[-1].to(DEVICE)
                loss = (per_sample_loss * batch_w).mean()
            else:
                loss = per_sample_loss.mean()

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())

        val_metrics, _, _ = evaluate_model(model, val_loader, title=f"{title} validation epoch {epoch:02d}")
        print(f"{title} epoch {epoch:02d} train_loss={np.mean(losses):.4f} val_f1={val_metrics['f1']:.4f}")

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1

        if stale >= patience:
            print("Early stopping")
            break

    model.load_state_dict(best_state)
    test_metrics, y_true, y_pred = evaluate_model(model, test_loader, title=f"{title} test")
    return model, test_metrics, y_true, y_pred


# 1. Load ERM Transformer

The ERM Transformer is the starting point for JTT and DFR. If `vanilla_transformer_subject_independent.pt` exists, we load it. Otherwise, we train a fresh ERM model and save it.


In [ ]:
USE_PRETRAINED_ERM = True
ERM_CHECKPOINT_PATH = "vanilla_transformer_subject_independent.pt"
FALLBACK_ERM_CHECKPOINT_PATH = "dfr_erm_transformer.pt"

base_train_loader = make_loader(X_train, y_train, batch_size=128, shuffle=True)
base_val_loader = make_loader(X_val, y_val, batch_size=128, shuffle=False)
base_test_loader = make_loader(X_test, y_test, batch_size=128, shuffle=False)

checkpoint_to_load = None
if USE_PRETRAINED_ERM and os.path.exists(ERM_CHECKPOINT_PATH):
    checkpoint_to_load = ERM_CHECKPOINT_PATH
elif USE_PRETRAINED_ERM and os.path.exists(FALLBACK_ERM_CHECKPOINT_PATH):
    checkpoint_to_load = FALLBACK_ERM_CHECKPOINT_PATH

if checkpoint_to_load is not None:
    print("Loading ERM Transformer from:", checkpoint_to_load)
    erm_model, erm_checkpoint = load_transformer_checkpoint(checkpoint_to_load)
    erm_metrics, erm_y_true, erm_y_pred = evaluate_model(erm_model, base_test_loader, title="Loaded ERM Transformer test")
else:
    print("No ERM checkpoint found; training ERM Transformer from scratch.")
    erm_model, erm_metrics, erm_y_true, erm_y_pred = train_erm(
        base_train_loader,
        base_val_loader,
        base_test_loader,
        title="ERM Transformer",
        epochs=30,
        patience=7,
    )
    save_checkpoint("dfr_erm_transformer.pt", erm_model, erm_metrics, method="erm")


Loading ERM Transformer from: vanilla_transformer_subject_independent.pt


/tmp/ipykernel_3218/4132389826.py:29: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)



Loaded ERM Transformer test
accuracy: 0.4503
precision: 0.4288
recall: 0.5220
f1: 0.4085
              precision    recall  f1-score   support

          AD     0.6558    0.4721    0.5490      8847
          CN     0.2591    0.8733    0.3997      1737
         FTD     0.3713    0.2205    0.2767      4036

    accuracy                         0.4503     14620
   macro avg     0.4288    0.5220    0.4085     14620
weighted avg     0.5302    0.4503    0.4561     14620



# 2. JTT

JTT does not require group labels.

Pipeline:

```text
1. Use the ERM model to find misclassified training samples.
2. Upweight those samples.
3. Train a second model with weighted ERM.
```


ERM model evaluated on the training split


In [ ]:
_, train_true, train_pred = evaluate_model(erm_model, make_loader(X_train, y_train, batch_size=128, shuffle=False), title="ERM train predictions for JTT")
train_error = train_true != train_pred

jtt_upweight = 5.0
jtt_weights = np.ones(len(y_train), dtype=np.float32)
jtt_weights[train_error] = jtt_upweight

print("Misclassified train samples:", int(train_error.sum()), "of", len(train_error))
print("JTT upweight factor:", jtt_upweight)



ERM train predictions for JTT
accuracy: 0.7498
precision: 0.7475
recall: 0.7134
f1: 0.7205
              precision    recall  f1-score   support

          AD     0.7397    0.7796    0.7591     13863
          CN     0.7592    0.8689    0.8103     17302
         FTD     0.7437    0.4917    0.5920      9588

    accuracy                         0.7498     40753
   macro avg     0.7475    0.7134    0.7205     40753
weighted avg     0.7489    0.7498    0.7415     40753

Misclassified train samples: 10197 of 40753
JTT upweight factor: 5.0


In [ ]:
jtt_train_loader = make_loader(X_train, y_train, weight=jtt_weights, batch_size=128, shuffle=True)
jtt_model, jtt_metrics, _, _ = train_erm(
    jtt_train_loader,
    base_val_loader,
    base_test_loader,
    title="JTT second-stage Transformer",
    epochs=30,
    patience=7,
    weighted=True,
)
save_checkpoint("jtt_transformer.pt", jtt_model, jtt_metrics, method="jtt")


/tmp/ipykernel_3218/4132389826.py:29: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)



JTT second-stage Transformer validation epoch 01
accuracy: 0.2485
precision: 0.3744
recall: 0.3504
f1: 0.2019
              precision    recall  f1-score   support

          AD     0.4850    0.1107    0.1802      6407
          CN     0.4251    0.0449    0.0812      5059
         FTD     0.2130    0.8958    0.3442      2955

    accuracy                         0.2485     14421
   macro avg     0.3744    0.3504    0.2019     14421
weighted avg     0.4082    0.2485    0.1791     14421

JTT second-stage Transformer epoch 01 train_loss=2.2050 val_f1=0.2019

JTT second-stage Transformer validation epoch 02
accuracy: 0.2774
precision: 0.3589
recall: 0.3524
f1: 0.2569
              precision    recall  f1-score   support

          AD     0.4865    0.1912    0.2745      6407
          CN     0.3760    0.1028    0.1614      5059
         FTD     0.2144    0.7631    0.3347      2955

    accuracy                         0.2774     14421
   macro avg     0.3589    0.3524    0.2569     14421
w

# 3. GroupDRO With Oracle Groups

GroupDRO uses group labels during training. Here we use oracle groups:

```text
group = diagnosis x subject
```

This is supervised/oracle group information, so it is not directly deployable if subject-group labels are unavailable, but it is useful as a strong reference baseline.


In [ ]:
def make_joint_keys(a, b):
    return np.array([f"{x}__{y}" for x, y in zip(a, b)])


def fit_group_encoder_on_train(train_keys):
    encoder = LabelEncoder()
    train_group = encoder.fit_transform(train_keys)
    mapping = {key: idx for idx, key in enumerate(encoder.classes_)}
    return train_group, encoder, mapping


def transform_groups_with_unknown(keys, mapping, unknown_value=0):
    return np.array([mapping.get(key, unknown_value) for key in keys], dtype=np.int64)

oracle_train_keys = make_joint_keys(y_train, subjects_train)
oracle_val_keys = make_joint_keys(y_val, subjects_val)
oracle_test_keys = make_joint_keys(y_test, subjects_test)

oracle_group_train, oracle_group_encoder, oracle_group_mapping = fit_group_encoder_on_train(oracle_train_keys)
oracle_group_val = transform_groups_with_unknown(oracle_val_keys, oracle_group_mapping)
oracle_group_test = transform_groups_with_unknown(oracle_test_keys, oracle_group_mapping)
oracle_n_groups = len(oracle_group_encoder.classes_)

print("Oracle groups:", oracle_n_groups)


Oracle groups: 52


In [ ]:
def train_group_dro(train_loader, val_loader, test_loader, n_groups, title="GroupDRO", epochs=30, lr=1e-4, patience=7, dro_eta=0.05, weight_decay=1e-4):
    model = new_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    q = torch.ones(n_groups, device=DEVICE) / n_groups

    best_val_f1 = -np.inf
    best_state = None
    stale = 0

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for batch_x, batch_y, batch_g in train_loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            batch_g = batch_g.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(batch_x)
            per_sample_loss = nn.functional.cross_entropy(logits, batch_y, reduction="none")

            group_losses = torch.zeros(n_groups, device=DEVICE)
            present = torch.zeros(n_groups, device=DEVICE)
            for g in batch_g.unique():
                mask = batch_g == g
                group_losses[g] = per_sample_loss[mask].mean()
                present[g] = 1.0

            with torch.no_grad():
                q = q * torch.exp(dro_eta * group_losses.detach() * present)
                q = q / q.sum().clamp_min(1e-12)

            loss = (q * group_losses).sum()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())

        val_metrics, _, _ = evaluate_model(model, val_loader, title=f"{title} validation epoch {epoch:02d}")
        print(f"{title} epoch {epoch:02d} train_loss={np.mean(losses):.4f} val_f1={val_metrics['f1']:.4f}")

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1

        if stale >= patience:
            print("Early stopping")
            break

    model.load_state_dict(best_state)
    test_metrics, y_true, y_pred = evaluate_model(model, test_loader, title=f"{title} test")
    return model, test_metrics, y_true, y_pred


In [ ]:
oracle_train_loader = make_loader(X_train, y_train, group=oracle_group_train, batch_size=128, shuffle=True)
oracle_val_loader = make_loader(X_val, y_val, group=oracle_group_val, batch_size=128, shuffle=False)
oracle_test_loader = make_loader(X_test, y_test, group=oracle_group_test, batch_size=128, shuffle=False)

oracle_groupdro_model, oracle_groupdro_metrics, _, _ = train_group_dro(
    oracle_train_loader,
    oracle_val_loader,
    oracle_test_loader,
    n_groups=oracle_n_groups,
    title="Oracle GroupDRO diagnosis x subject",
    epochs=30,
    patience=7,
)
save_checkpoint("oracle_groupdro_diagnosis_subject.pt", oracle_groupdro_model, oracle_groupdro_metrics, method="oracle_groupdro_diagnosis_subject")


/tmp/ipykernel_3218/4132389826.py:29: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)



Oracle GroupDRO diagnosis x subject validation epoch 01
accuracy: 0.4078
precision: 0.3889
recall: 0.3915
f1: 0.3876
              precision    recall  f1-score   support

          AD     0.4971    0.4884    0.4927      6407
          CN     0.4073    0.3445    0.3733      5059
         FTD     0.2623    0.3415    0.2967      2955

    accuracy                         0.4078     14421
   macro avg     0.3889    0.3915    0.3876     14421
weighted avg     0.4175    0.4078    0.4106     14421

Oracle GroupDRO diagnosis x subject epoch 01 train_loss=1.0067 val_f1=0.3876

Oracle GroupDRO diagnosis x subject validation epoch 02
accuracy: 0.4063
precision: 0.3829
recall: 0.3709
f1: 0.3687
              precision    recall  f1-score   support

          AD     0.4785    0.5636    0.5176      6407
          CN     0.4660    0.2971    0.3629      5059
         FTD     0.2041    0.2521    0.2256      2955

    accuracy                         0.4063     14421
   macro avg     0.3829    0.3709 

# 4. DFR / Last-Layer Retraining

DFR asks whether the ERM model already learned useful disease features, but the final classifier overweights shortcut features.

We run two DFR variants:

```text
DFR-oracle:
    reweighting set balanced by diagnosis x subject
    uses subject/group labels

DFR-class-balanced:
    reweighting set balanced by diagnosis only
    does not use subject/group labels
```

Then we run a small-data ablation:

```text
k samples per diagnosis x subject group
```

to test whether DFR works with a small balanced validation set.


In [ ]:
def make_balanced_reweighting_indices(y, groups=None, max_per_group=None, random_state=42):
    rng = np.random.default_rng(random_state)

    if groups is None:
        keys = np.asarray(y).astype(str)
    else:
        keys = make_joint_keys(y, groups)

    group_to_indices = {}
    for idx, key in enumerate(keys):
        group_to_indices.setdefault(key, []).append(idx)

    if max_per_group is None:
        max_per_group = min(len(v) for v in group_to_indices.values())

    selected = []
    for _, idxs in group_to_indices.items():
        idxs = np.array(idxs)
        if len(idxs) > max_per_group:
            idxs = rng.choice(idxs, size=max_per_group, replace=False)
        selected.extend(idxs.tolist())

    selected = np.array(selected, dtype=np.int64)
    rng.shuffle(selected)
    return selected


def freeze_feature_extractor_for_dfr(model):
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True


def reset_classifier(model):
    n_features = model.classifier.in_features
    n_classes = model.classifier.out_features
    model.classifier = nn.Linear(n_features, n_classes).to(DEVICE)


In [ ]:
def train_dfr_head(base_model, X_reweight, y_reweight, X_eval_val, y_eval_val, X_eval_test, y_eval_test, epochs=100, lr=1e-3, patience=10, batch_size=128, title="DFR"):
    model = new_model()
    model.load_state_dict(base_model.state_dict())
    reset_classifier(model)
    freeze_feature_extractor_for_dfr(model)

    train_loader = make_loader(X_reweight, y_reweight, batch_size=batch_size, shuffle=True)
    val_loader = make_loader(X_eval_val, y_eval_val, batch_size=batch_size, shuffle=False)
    test_loader = make_loader(X_eval_test, y_eval_test, batch_size=batch_size, shuffle=False)

    optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=lr, weight_decay=1e-3)
    best_val_f1 = -np.inf
    best_state = None
    stale = 0

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        val_metrics, _, _ = evaluate_model(model, val_loader, title=f"{title} validation epoch {epoch:02d}")
        print(f"{title} epoch {epoch:02d} train_loss={np.mean(losses):.4f} val_f1={val_metrics['f1']:.4f}")

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1

        if stale >= patience:
            print("Early stopping")
            break

    model.load_state_dict(best_state)
    test_metrics, y_true, y_pred = evaluate_model(model, test_loader, title=f"{title} test")
    return model, test_metrics, y_true, y_pred


In [ ]:
# DFR-oracle: balanced by diagnosis x subject.
dfr_oracle_idx = make_balanced_reweighting_indices(
    y_val,
    groups=subjects_val,
    max_per_group=None,
    random_state=RANDOM_STATE,
)

print("DFR-oracle reweighting samples:", len(dfr_oracle_idx))
print("DFR-oracle groups:", len(np.unique(make_joint_keys(y_val[dfr_oracle_idx], subjects_val[dfr_oracle_idx]))))

dfr_oracle_model, dfr_oracle_metrics, _, _ = train_dfr_head(
    erm_model,
    X_val[dfr_oracle_idx],
    y_val[dfr_oracle_idx],
    X_val,
    y_val,
    X_test,
    y_test,
    epochs=100,
    lr=1e-3,
    patience=10,
    batch_size=128,
    title="DFR-oracle diagnosis x subject",
)
save_checkpoint("dfr_oracle_diagnosis_subject.pt", dfr_oracle_model, dfr_oracle_metrics, method="dfr_oracle_diagnosis_subject")


DFR-oracle reweighting samples: 10314
DFR-oracle groups: 18


/tmp/ipykernel_3218/4132389826.py:29: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)



DFR-oracle diagnosis x subject validation epoch 01
accuracy: 0.5380
precision: 0.5058
recall: 0.4914
f1: 0.4940
              precision    recall  f1-score   support

          AD     0.5755    0.6627    0.6160      6407
          CN     0.5380    0.5294    0.5336      5059
         FTD     0.4039    0.2822    0.3323      2955

    accuracy                         0.5380     14421
   macro avg     0.5058    0.4914    0.4940     14421
weighted avg     0.5272    0.5380    0.5290     14421

DFR-oracle diagnosis x subject epoch 01 train_loss=0.9815 val_f1=0.4940

DFR-oracle diagnosis x subject validation epoch 02
accuracy: 0.5576
precision: 0.5312
recall: 0.5201
f1: 0.5229
              precision    recall  f1-score   support

          AD     0.6161    0.6390    0.6273      6407
          CN     0.5338    0.5819    0.5568      5059
         FTD     0.4436    0.3394    0.3846      2955

    accuracy                         0.5576     14421
   macro avg     0.5312    0.5201    0.5229     1

In [ ]:
 dfr_class_idx = make_balanced_reweighting_indices(
    y_val,
    groups=None,
    max_per_group=None,
    random_state=RANDOM_STATE,
)

print("DFR-class-balanced reweighting samples:", len(dfr_class_idx))
print("DFR-class-balanced class counts:", np.bincount(y_val[dfr_class_idx], minlength=len(class_names)))

dfr_class_model, dfr_class_metrics, _, _ = train_dfr_head(
    erm_model,
    X_val[dfr_class_idx],
    y_val[dfr_class_idx],
    X_val,
    y_val,
    X_test,
    y_test,
    epochs=100,
    lr=1e-3,
    patience=10,
    batch_size=128,
    title="DFR class-balanced",
)
save_checkpoint("dfr_class_balanced.pt", dfr_class_model, dfr_class_metrics, method="dfr_class_balanced")


DFR-class-balanced reweighting samples: 8865
DFR-class-balanced class counts: [2955 2955 2955]


/tmp/ipykernel_3218/4132389826.py:29: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)



DFR class-balanced validation epoch 01
accuracy: 0.5226
precision: 0.5224
recall: 0.5332
f1: 0.5148
              precision    recall  f1-score   support

          AD     0.6633    0.4673    0.5483      6407
          CN     0.5409    0.5683    0.5543      5059
         FTD     0.3630    0.5641    0.4418      2955

    accuracy                         0.5226     14421
   macro avg     0.5224    0.5332    0.5148     14421
weighted avg     0.5588    0.5226    0.5286     14421

DFR class-balanced epoch 01 train_loss=1.0248 val_f1=0.5148

DFR class-balanced validation epoch 02
accuracy: 0.5429
precision: 0.5527
recall: 0.5690
f1: 0.5392
              precision    recall  f1-score   support

          AD     0.7034    0.4461    0.5459      6407
          CN     0.5612    0.5918    0.5761      5059
         FTD     0.3936    0.6690    0.4956      2955

    accuracy                         0.5429     14421
   macro avg     0.5527    0.5690    0.5392     14421
weighted avg     0.5900    0.54